In [1]:
import subprocess
import os

# 1. Install python3.10 and venv support on Colab
subprocess.run(["sudo", "apt-get", "update", "-y"], check=True)
subprocess.run(["sudo", "apt-get", "install", "python3.10", "python3.10-venv", "python3.10-dev", "-y"], check=True)

# 2. Create the virtual environment
subprocess.run(["python3.10", "-m", "venv", "/content/venv"], check=True)

# 3. Upgrade pip inside the venv
subprocess.run(["/content/venv/bin/python", "-m", "pip", "install", "--upgrade", "pip"], check=True)

CompletedProcess(args=['/content/venv/bin/python', '-m', 'pip', 'install', '--upgrade', 'pip'], returncode=0)

In [2]:
# 1. Install python3.10 and venv support
!sudo apt-get update -y
!sudo apt-get install python3.10 python3.10-venv python3.10-dev -y

# 2. Create an isolated virtual environment
!python3.10 -m venv /content/venv

# 3. Upgrade pip inside the virtual environment
!/content/venv/bin/python -m pip install --upgrade pip

# 4. Install the required serving pins
!/content/venv/bin/python -m pip install \
    "vllm==0.6.*" \
    "transformers==4.46.*" \
    "accelerate==1.1.*" \
    "httpx==0.27.*" \
    "openai==1.54.*"

print("Virtual environment ready with vLLM installed!")

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [3]:

# # Week 3 shared Colab scaffold
#
# This file is the source of truth for the reusable Colab cells every week-3 lab
# uses. It is written in py-percent format: each `# %%` block is one standalone,
# pasteable Colab cell. Copy the cells you need into the day's notebook in the
# order the lab README gives.
#
# Why this scaffold exists: a Colab notebook runs cells one at a time, top to
# bottom, and a cell blocks until it returns. A live inference server does not
# return: it runs until you kill it. So you cannot "start the server" in one cell
# and "watch it" in the next the way you would in a terminal with two panes. The
# pattern here is launch-then-poll: one cell launches the server as a background
# subprocess and returns immediately, and a second cell polls the health endpoint
# until the server answers or a timeout fires. Every long-running piece (the
# server, the nvidia-smi sampler) runs in the background and is watched by a
# short cell that returns.
#
# Pin source: versions come from ../../../PINS.md (course root). The vLLM-on-T4 pin
# is verified on a real free-tier T4 before the cohort starts; the confirmed
# version and date land in PINS.md under "Verification status". Do not invent a
# vLLM version here; read the pin.
#
# Convert to a .ipynb when you want a notebook file (the .py stays the source of
# truth):
#   uvx jupytext --to ipynb colab_scaffold.py

# %%
# PINS block. These mirror ../../../PINS.md (course root, the single source of
# truth). If a pin changes, it changes in PINS.md first, then here. The vLLM pin
# is the load-bearing one: it must be the version confirmed on a real free-tier
# T4 during the pre-cohort verification pass. Read PINS.md before you run this.
#
# PINS (from ../../../PINS.md):
#   VLLM_PIN=0.6.*          # OpenAI server; runs the xformers backend on sm75
#   BITSANDBYTES_PIN=0.49.2 # int8/int4 load path (day 1 profiling); 0.44.* is
#                           # broken on Colab's cu128 torch, see PINS.md
#   AUTOAWQ_PIN=0.2.*       # AWQ weights load path (day 4)
#   TRANSFORMERS_PIN=4.46.* # streaming generation (day 2)
#   ACCELERATE_PIN=1.1.*    # device placement
#   HTTPX_PIN=0.27.*        # async A/B client (day 3)
#   OPENAI_PIN=1.54.*       # the client that proves the /v1 contract

# %%
# Cell: the pins and the installer function. Defines only, installs nothing.
# Paste this on every week-3 day. Then paste ONE of the two install cells below,
# whichever the day's README names. Day 1 profiles with transformers and must
# NOT install vLLM; days 2 to 5 serve, and must.
import subprocess, sys

# Pins mirrored from ../../../PINS.md. Keep these two in sync (PINS.md wins).
VLLM_PIN = "0.6.*"
BITSANDBYTES_PIN = "0.49.2"
AUTOAWQ_PIN = "0.2.*"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"
HTTPX_PIN = "0.27.*"
OPENAI_PIN = "1.54.*"

def pip_install(*specs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *specs]
    print("installing:", " ".join(specs))
    subprocess.run(cmd, check=True)

In [4]:
# %%
# INSTALL CELL B: the serving set. This is days 3 to 5 (day 2 is CELL A:
# direct transformers loads crash on vLLM's numpy - verified on T4 2026-08-07). About 30 minutes on a
# cold runtime, and it prints almost nothing for most of it, so start it and go
# and fill in your prediction card. vLLM brings its own torch; do NOT install a
# second one.
#
# transformers and accelerate are NOT optional here, even on days you never call
# them directly. vLLM 0.6.x installs its own torch (2.5.1), which downgrades
# Colab's torch and leaves Colab's preinstalled torchaudio compiled against the
# wrong ABI. Colab's preinstalled transformers imports torchaudio at module load,
# so vLLM then dies during startup with
#   OSError: _torchaudio.abi3.so: undefined symbol: aoti_torch_abi_version
# Pinning transformers to 4.46 removes that import path. Verified on a T4,
# 2026-07-27: without these two lines the server never comes up.
#
# autoawq is only needed on day 4; that README says so and adds it to this call.
PYTHON_BIN = "/content/venv/bin/python"

VLLM_PIN = "0.6.*"
BITSANDBYTES_PIN = "0.49.2"
AUTOAWQ_PIN = "0.2.*"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"
HTTPX_PIN = "0.27.*"
OPENAI_PIN = "1.54.*"

!{PYTHON_BIN} -m pip install \
    "vllm=={VLLM_PIN}" \
    "transformers=={TRANSFORMERS_PIN}" \
    "accelerate=={ACCELERATE_PIN}" \
    "autoawq=={AUTOAWQ_PIN}" \
    "httpx=={HTTPX_PIN}" \
    "openai=={OPENAI_PIN}"


# NOTE (2026-08-07, verified the hard way on a live T4): do NOT add a
# numpy>=2 pin here - vLLM 0.6.x requires numpy<2 and the install fails
# outright. CELL B as verified 2026-07-27 runs on the numpy vLLM chooses.
print("serving pins installed")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 18.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 58.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 122.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 127.4 MB/s  0:00:00
  Created wheel for autoawq: filename=autoawq-0.2.9-py3-none-any.whl size=115198 sha256=2f573be33a3da9fd59788d75f512523661291f72cc555d6fa8f408bd28f0ca4a
  Stored in directory: /root/.cache/pip/wheels/13/2d/f6/3161d1c3acde652ce9b205d1c70eba821339e6d9a92dd81e12
Successfully built autoawq
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2026.7.0
    Uninstalling fsspec-2026.7.0:
      Successfully uninstalled fsspec-2026.7.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [autoawq]
serving pins installed


In [5]:
import os
import subprocess

PYTHON_BIN = "/content/venv/bin/python"

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
PORT = 8000
SERVER_LOG = "/content/server.log"

SERVER_ARGS = {
    "--model": "Qwen/Qwen2.5-1.5B-Instruct-AWQ",
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": "8000",
    "--quantization": "awq",
    "--enable-auto-tool-choice": None,
    "--tool-call-parser": "hermes",
}

def build_cmd(args: dict) -> list:
    cmd = [PYTHON_BIN, "-m", "vllm.entrypoints.openai.api_server"]
    for k, v in args.items():
        if v is None:
            cmd.append(k)
        else:
            cmd += [k, str(v)]
    return cmd

def launch_server(args: dict = None):
    args = SERVER_ARGS if args is None else args
    cmd = build_cmd(args)

    print("launching:", " ".join(cmd))

    logf = open(SERVER_LOG, "wb")

    proc = subprocess.Popen(
        cmd,
        stdout=logf,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

    print(f"server pid {proc.pid}, logging to {SERVER_LOG}")
    return proc

server = launch_server(SERVER_ARGS)

launching: /content/venv/bin/python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct-AWQ --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000 --quantization awq --enable-auto-tool-choice --tool-call-parser hermes
server pid 7729, logging to /content/server.log


In [6]:
!tail -n 30 /content/server.log

In [7]:
!fuser -v 8000/tcp

In [8]:
!curl -s http://localhost:8000/v1/models

In [9]:
# %%
# Cell: health poll.
# The launch cell returned immediately; the server is still loading weights in
# the background. This cell polls GET /v1/models until it answers 200 or the
# timeout fires. First launch on a fresh runtime downloads the model, so the
# first poll can take a while; that is what the 300s timeout is for. On timeout
# it prints the last 30 log lines so you can see why (usually still downloading,
# or an OOM, or a bad flag).
import time, urllib.request, urllib.error

def tail_log(path=SERVER_LOG, n=30):
    try:
        with open(path, "r", errors="replace") as fh:
            lines = fh.readlines()
        return "".join(lines[-n:])
    except FileNotFoundError:
        return "(no log file yet)"

def wait_for_health(port=PORT, timeout_s=300, interval_s=3):
    url = f"http://localhost:{port}/v1/models"
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    waited = int(timeout_s - (deadline - time.time()))
                    print(f"server healthy after about {waited}s: {url} -> 200")
                    return True
        except (urllib.error.URLError, ConnectionError, OSError):
            pass  # not up yet
        time.sleep(interval_s)
    print(f"TIMED OUT after {timeout_s}s waiting for {url}")
    print("last 30 log lines:")
    print(tail_log())
    print("server did not come up. common causes: model still downloading "
          "(rerun this cell), OOM at load (lower --gpu-memory-utilization to "
          "0.80), or a bad flag (bf16 on sm75; use --dtype half).")
    return False

healthy = wait_for_health()

server healthy after about 39s: http://localhost:8000/v1/models -> 200


In [11]:
%%writefile bench.py


"""Benchmark harness for the serving stack (week 3 day 5 reference).

Sweeps concurrency levels against an OpenAI-compatible endpoint and reports the
numbers the week-3 lab is graded on: tokens/sec, TTFT p50/p95, end-to-end
latency p95, and error counts per level.

CLI contract (the week-3 lab is written against this exactly):

    python bench.py \
        --base-url http://localhost:8000 \
        --model Qwen/Qwen2.5-0.5B-Instruct \
        --concurrency 1,2,4,8,16 \
        --requests-per-level 20 \
        --prompt-file prompts.txt \
        --out bench_report.json

Method:
  - Every request uses stream=true so TTFT is the wall-clock time to the first
    SSE content chunk. End-to-end latency is time to the [DONE] sentinel.
  - One warm-up request per level is fired and excluded from the statistics, so
    a cold cache or JIT does not skew the first measured level.
  - Errors are counted per level and never crash the sweep; a level with all
    errors still reports (with null latencies).
  - Output is written to --out as JSON and printed as a readable table. Re-runs
    append to a `runs` array in the same file when it already exists, so an A/B
    (CPU vs vLLM, Monday vs Wednesday) accumulates in one place.

Dependencies: httpx (async). Pin per ../../PINS.md.
"""

from __future__ import annotations

import argparse
import asyncio
import json
import os
import statistics
import time
from dataclasses import dataclass, field
from typing import Optional

import httpx


# --------------------------------------------------------------------------- #
# Per-request measurement                                                      #
# --------------------------------------------------------------------------- #

@dataclass
class RequestResult:
    ok: bool
    ttft_s: Optional[float] = None          # time to first content chunk
    latency_s: Optional[float] = None       # time to [DONE]
    completion_tokens: int = 0              # counted from streamed chunks
    error: Optional[str] = None


async def _one_request(
    client: httpx.AsyncClient,
    base_url: str,
    model: str,
    prompt: str,
    max_tokens: int,
) -> RequestResult:
    """Fire one streaming completion and measure TTFT and end-to-end latency.

    TTFT is the time from send to the first SSE frame that carries non-empty
    content (the role-announcement frame and empty deltas do not count).
    """
    body = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "stream": True,
        "temperature": 0.0,
    }
    url = base_url.rstrip("/") + "/v1/chat/completions"
    start = time.perf_counter()
    ttft: Optional[float] = None
    tokens = 0

    try:
        async with client.stream("POST", url, json=body) as response:
            if response.status_code != 200:
                # Drain so the connection can be reused, then report the error.
                text = (await response.aread()).decode("utf-8", "replace")[:200]
                return RequestResult(
                    ok=False, error=f"HTTP {response.status_code}: {text}"
                )
            async for line in response.aiter_lines():
                if not line or not line.startswith("data: "):
                    continue
                data = line[len("data: "):]
                if data == "[DONE]":
                    break
                try:
                    chunk = json.loads(data)
                except json.JSONDecodeError:
                    continue
                delta = chunk.get("choices", [{}])[0].get("delta", {})
                content = delta.get("content")
                if content:
                    if ttft is None:
                        ttft = time.perf_counter() - start
                    tokens += 1
        latency = time.perf_counter() - start
        return RequestResult(
            ok=True, ttft_s=ttft, latency_s=latency, completion_tokens=tokens
        )
    except Exception as exc:  # network error, timeout, reset: count, do not raise
        return RequestResult(ok=False, error=f"{type(exc).__name__}: {exc}")


# --------------------------------------------------------------------------- #
# Per-level sweep                                                              #
# --------------------------------------------------------------------------- #

@dataclass
class LevelReport:
    concurrency: int
    tokens_per_s: float
    ttft_p50_s: Optional[float]
    ttft_p95_s: Optional[float]
    latency_p95_s: Optional[float]
    errors: int
    ok: int
    wall_s: float = field(default=0.0)


def _percentile(values: list[float], pct: float) -> Optional[float]:
    """Nearest-rank percentile, robust for tiny samples."""
    if not values:
        return None
    ordered = sorted(values)
    if len(ordered) == 1:
        return round(ordered[0], 4)
    rank = max(1, int(round(pct / 100.0 * len(ordered))))
    rank = min(rank, len(ordered))
    return round(ordered[rank - 1], 4)


async def _run_level(
    client: httpx.AsyncClient,
    base_url: str,
    model: str,
    prompts: list[str],
    concurrency: int,
    requests_per_level: int,
    max_tokens: int,
) -> LevelReport:
    """Run one concurrency level: one warm-up (excluded), then the measured batch."""
    # Warm-up, excluded from stats. Ignore its result entirely.
    await _one_request(client, base_url, model, prompts[0], max_tokens)

    # Fire `requests_per_level` requests `concurrency` at a time. A semaphore
    # caps in-flight requests so the level actually holds the intended load.
    semaphore = asyncio.Semaphore(concurrency)

    async def _guarded(index: int) -> RequestResult:
        async with semaphore:
            prompt = prompts[index % len(prompts)]
            return await _one_request(client, base_url, model, prompt, max_tokens)

    level_start = time.perf_counter()
    results = await asyncio.gather(
        *(_guarded(i) for i in range(requests_per_level))
    )
    wall = time.perf_counter() - level_start

    ok = [r for r in results if r.ok]
    errors = len(results) - len(ok)
    ttfts = [r.ttft_s for r in ok if r.ttft_s is not None]
    latencies = [r.latency_s for r in ok if r.latency_s is not None]
    total_tokens = sum(r.completion_tokens for r in ok)

    # System throughput: total completion tokens over the level wall-clock, so
    # it rises with concurrency until the server saturates.
    tokens_per_s = round(total_tokens / wall, 2) if wall > 0 else 0.0

    return LevelReport(
        concurrency=concurrency,
        tokens_per_s=tokens_per_s,
        ttft_p50_s=_percentile(ttfts, 50),
        ttft_p95_s=_percentile(ttfts, 95),
        latency_p95_s=_percentile(latencies, 95),
        errors=errors,
        ok=len(ok),
        wall_s=round(wall, 3),
    )


# --------------------------------------------------------------------------- #
# Orchestration                                                                #
# --------------------------------------------------------------------------- #

def _load_prompts(path: str) -> list[str]:
    with open(path, encoding="utf-8") as handle:
        prompts = [line.strip() for line in handle if line.strip()]
    if not prompts:
        raise SystemExit(f"prompt file {path!r} has no non-empty lines")
    return prompts


def _print_table(levels: list[LevelReport]) -> None:
    header = (
        f"{'conc':>4}  {'tok/s':>8}  {'ttft_p50':>9}  {'ttft_p95':>9}  "
        f"{'lat_p95':>8}  {'ok':>4}  {'err':>4}"
    )
    print(header)
    print("-" * len(header))
    for lv in levels:
        def fmt(value: Optional[float]) -> str:
            return f"{value:.3f}" if value is not None else "  n/a"
        print(
            f"{lv.concurrency:>4}  {lv.tokens_per_s:>8.2f}  "
            f"{fmt(lv.ttft_p50_s):>9}  {fmt(lv.ttft_p95_s):>9}  "
            f"{fmt(lv.latency_p95_s):>8}  {lv.ok:>4}  {lv.errors:>4}"
        )


def _write_report(out_path: str, run_record: dict) -> None:
    """Append this run to `runs` in the JSON file, creating it if absent."""
    document: dict = {"runs": []}
    if os.path.exists(out_path):
        try:
            with open(out_path, encoding="utf-8") as handle:
                existing = json.load(handle)
            if isinstance(existing, dict) and isinstance(existing.get("runs"), list):
                document = existing
        except (json.JSONDecodeError, OSError):
            # Corrupt or unreadable prior file: start fresh rather than crash.
            document = {"runs": []}
    document["runs"].append(run_record)
    with open(out_path, "w", encoding="utf-8") as handle:
        json.dump(document, handle, indent=2)


async def _sweep(args: argparse.Namespace) -> tuple[dict, list[LevelReport]]:
    prompts = _load_prompts(args.prompt_file)
    concurrency_levels = [int(c) for c in args.concurrency.split(",") if c.strip()]

    timeout = httpx.Timeout(args.timeout, connect=10.0)
    limits = httpx.Limits(max_connections=max(concurrency_levels) + 4)
    levels: list[LevelReport] = []

    headers = {}
    if getattr(args, "api_key", ""):
        headers["Authorization"] = "Bearer " + args.api_key
    async with httpx.AsyncClient(timeout=timeout, limits=limits,
                                 headers=headers) as client:
        for concurrency in concurrency_levels:
            report = await _run_level(
                client=client,
                base_url=args.base_url,
                model=args.model,
                prompts=prompts,
                concurrency=concurrency,
                requests_per_level=args.requests_per_level,
                max_tokens=args.max_tokens,
            )
            levels.append(report)
            # Progress line per level so a long sweep is not silent.
            print(
                f"[level {concurrency}] tok/s={report.tokens_per_s} "
                f"ttft_p95={report.ttft_p95_s} errors={report.errors}",
                flush=True,
            )

    return {
        "timestamp": int(time.time()),
        "base_url": args.base_url,
        "model": args.model,
        "requests_per_level": args.requests_per_level,
        "max_tokens": args.max_tokens,
        "prompt_file": args.prompt_file,
        "levels": [vars(lv) for lv in levels],
    }, levels


def main() -> None:
    parser = argparse.ArgumentParser(description="serving-stack benchmark harness")
    parser.add_argument("--base-url", default="http://localhost:8000")
    parser.add_argument("--model", required=True)
    parser.add_argument("--concurrency", default="1,2,4,8,16",
                        help="comma-separated concurrency levels")
    parser.add_argument("--requests-per-level", type=int, default=20)
    parser.add_argument("--prompt-file", default="prompts.sample.txt")
    parser.add_argument("--out", default="bench_report.json")
    parser.add_argument("--max-tokens", type=int, default=128)
    parser.add_argument("--api-key", default=os.environ.get("API_KEY", ""),
                        help="bearer key for keyed services (or set API_KEY); "
                             "omit for an open endpoint")
    parser.add_argument("--timeout", type=float, default=120.0,
                        help="per-request timeout in seconds")
    args = parser.parse_args()

    run_record, levels = asyncio.run(_sweep(args))
    print()
    _print_table(levels)
    _write_report(args.out, run_record)
    print(f"\nwrote {args.out} (run appended)")


if __name__ == "__main__":
    main()

Writing bench.py


In [43]:
!python bench.py \
  --base-url http://localhost:8000 \
  --model Qwen/Qwen2.5-1.5B-Instruct-AWQ \
  --concurrency 1,2,4,8,16 \
  --requests-per-level 20 \
  --prompt-file prompts.txt \
  --out bench_report.json

[level 1] tok/s=78.62 ttft_p95=0.0888 errors=0
[level 2] tok/s=158.67 ttft_p95=0.0919 errors=0
[level 4] tok/s=275.39 ttft_p95=0.1214 errors=0
[level 8] tok/s=454.94 ttft_p95=0.2076 errors=0
[level 16] tok/s=683.82 ttft_p95=0.2499 errors=0

conc     tok/s   ttft_p50   ttft_p95   lat_p95    ok   err
----------------------------------------------------------
   1     78.62      0.053      0.089     1.629    20     0
   2    158.67      0.054      0.092     1.581    20     0
   4    275.39      0.062      0.121     1.751    20     0
   8    454.94      0.153      0.208     2.037    20     0
  16    683.82      0.246      0.250     2.431    20     0

wrote bench_report.json (run appended)


In [52]:
import json

levels = json.load(open("bench_report.json"))["runs"][-1]["levels"]

for L in levels:
    ttft = L["ttft_p95_s"]
    lat = L["latency_p95_s"]

    print(
        f"c={L['concurrency']:>2}  "
        f"tok/s={L['tokens_per_s']:>7.1f}  "
        f"ttft_p95={'N/A' if ttft is None else f'{ttft:.3f}'}  "
        f"lat_p95={'N/A' if lat is None else f'{lat:.3f}'}  "
        f"errors={L['errors']}"
    )

TARGET_P95_S = 2.0

under = [
    L for L in levels
    if L["errors"] == 0
    and L["latency_p95_s"] is not None
    and L["latency_p95_s"] <= TARGET_P95_S
]

knee = max(under, key=lambda L: L["concurrency"]) if under else None

print("knee:", knee)

c= 1  tok/s=   78.6  ttft_p95=0.089  lat_p95=1.629  errors=0
c= 2  tok/s=  158.7  ttft_p95=0.092  lat_p95=1.581  errors=0
c= 4  tok/s=  275.4  ttft_p95=0.121  lat_p95=1.751  errors=0
c= 8  tok/s=  454.9  ttft_p95=0.208  lat_p95=2.037  errors=0
c=16  tok/s=  683.8  ttft_p95=0.250  lat_p95=2.431  errors=0
knee: {'concurrency': 4, 'tokens_per_s': 275.39, 'ttft_p50_s': 0.0617, 'ttft_p95_s': 0.1214, 'latency_p95_s': 1.7514, 'errors': 0, 'ok': 20, 'wall_s': 7.273}


In [53]:
with open("knee.json", "w") as f:
    json.dump({
        "target_p95_s": TARGET_P95_S,
        "knee_concurrency": knee["concurrency"] if knee else None
    }, f)

In [54]:
%%writefile capacity-note.md
# Capacity note (team, one page)

## The numbers

- Locked model: Qwen/Qwen2.5-1.5B-Instruct-AWQ
- Target p95 end-to-end latency (your SLO today): 2.0 seconds
- Knee concurrency (highest concurrency whose p95 is still under target): 4
- Tokens per second at the knee: 275.39
- Max sustainable request rate at the target p95: 2.75 req/s

## The limiting family

- Memory-bound/decode-limited: throughput continues increasing as concurrency rises, but p95 latency crosses the 2.0-second SLO at concurrency 8, indicating that the decode-side memory-bandwidth limit is being approached.

## Why the knee, not the peak

- The knee is the highest concurrency that still meets the 2.0-second p95 SLO. Although peak throughput reaches 683.82 tokens/s at concurrency 16, its 2.4308-second p95 latency violates the SLO, so concurrency 4 is the appropriate operating point.

Overwriting capacity-note.md


In [55]:
# Green-check verifier for Lab W3D5 (benchmark harness).
# Paste this as the last cell of your day-5 notebook and run it. It reads
# bench_report.json (from the harness) and capacity-note.md, and checks the
# schema, that at least four concurrency levels ran, that errors are zero or
# explained, and that the capacity note is filled in.
#
# Last line is exactly one of:
#   GREEN CHECK: PASS
#   GREEN CHECK: FAIL (<reason>)
# No interactivity, no arguments; exit code matches.

import json, os, re

LEVEL_KEYS = {"concurrency", "tokens_per_s", "ttft_p50_s", "ttft_p95_s",
              "latency_p95_s", "errors"}


class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""


def fail(reason: str) -> "NoReturn":
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()


def main() -> None:
    # 1) bench report
    if not os.path.exists("bench_report.json"):
        fail("bench_report.json not found; run the harness in Cell 3")
    try:
        with open("bench_report.json") as fh:
            document = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"bench_report.json is not valid JSON: {exc}")

    # bench.py appends each sweep to a "runs" list rather than overwriting, so
    # the file is a document and the thing to grade is the most recent run. A
    # bare list is also accepted, for a report assembled by hand.
    if isinstance(document, dict) and isinstance(document.get("runs"), list):
        if not document["runs"]:
            fail("bench_report.json has no runs; the harness wrote nothing")
        levels = document["runs"][-1].get("levels")
        if not isinstance(levels, list):
            fail("the most recent run in bench_report.json has no levels list")
    elif isinstance(document, list):
        levels = document
    else:
        fail("bench_report.json must be the harness output ({'runs': [...]}) "
             "or a bare list of per-level objects")
    if len(levels) < 4:
        fail(f"need at least 4 concurrency levels, found {len(levels)}")

    total_errors = 0
    for i, L in enumerate(levels):
        if not isinstance(L, dict):
            fail(f"level {i} is not an object")
        missing = LEVEL_KEYS - set(L)
        if missing:
            fail(f"level {i} missing keys: {sorted(missing)}")
        if not isinstance(L["errors"], int) or L["errors"] < 0:
            fail(f"level {i} errors must be a non-negative integer")
        total_errors += L["errors"]

    # 2) the knee file from Cell 5
    if not os.path.exists("knee.json"):
        fail("knee.json not found; write it in Cell 5")
    try:
        with open("knee.json") as fh:
            knee = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"knee.json is not valid JSON: {exc}")
    target = knee.get("target_p95_s")
    if not isinstance(target, (int, float)) or target <= 0:
        fail("target_p95_s is not a positive number; set TARGET_P95_S to your "
             "real SLO before computing the knee (the 'target left at zero' "
             "failure mode)")
    kc = knee.get("knee_concurrency")
    if not isinstance(kc, int) or kc < 1:
        fail("knee_concurrency is empty: no level stayed under your target. "
             "Either your SLO is stricter than this stack can serve (explain "
             "that in the note) or the target was never set from the card")

    # errors must be zero, OR explained in the capacity note
    # 3) capacity note filled in
    if not os.path.exists("capacity-note.md"):
        fail("capacity-note.md not found")
    with open("capacity-note.md") as fh:
        note = fh.read()
    remaining = re.findall(r"FILL:", note)
    if remaining:
        fail(f"capacity-note.md has {len(remaining)} unfilled FILL: placeholders")

    if total_errors > 0 and not re.search(r"error", note, re.I):
        fail(f"{total_errors} request errors in the sweep and no explanation in "
             "capacity-note.md; zero errors, or explain them")

    # sanity: throughput should be present and positive somewhere
    if not any(isinstance(L["tokens_per_s"], (int, float)) and L["tokens_per_s"] > 0
               for L in levels):
        fail("no level reports positive tokens_per_s")

    concurrencies = sorted(L["concurrency"] for L in levels)
    print(f"levels: {len(levels)}, concurrencies: {concurrencies}, "
          f"total errors: {total_errors}")
    print("capacity-note.md: all fields filled")
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    # A notebook cell cannot exit nonzero without printing a red traceback over
    # the result line, so only signal by exit code when run as a plain script.
    try:
        get_ipython()  # defined only inside IPython/Colab
    except NameError:
        raise SystemExit(1)


levels: 5, concurrencies: [1, 2, 4, 8, 16], total errors: 0
capacity-note.md: all fields filled
GREEN CHECK: PASS


EXTRA LAB"

In [56]:
import json

try:
    levels = json.load(open("bench_report.json"))["runs"][-1]["levels"]
except FileNotFoundError:
    levels = [
        {"concurrency": 1,  "tokens_per_s": 38.2,  "latency_p95_s": 0.9,  "errors": 0},
        {"concurrency": 2,  "tokens_per_s": 71.5,  "latency_p95_s": 1.1,  "errors": 0},
        {"concurrency": 4,  "tokens_per_s": 128.4, "latency_p95_s": 1.4,  "errors": 0},
        {"concurrency": 8,  "tokens_per_s": 210.7, "latency_p95_s": 2.3,  "errors": 0},
        {"concurrency": 16, "tokens_per_s": 224.9, "latency_p95_s": 5.8,  "errors": 0},
    ]
    print("using the sample bench_report -- swap in your own file for a real answer")

for L in levels:
    print(L)

{'concurrency': 1, 'tokens_per_s': 78.62, 'ttft_p50_s': 0.0532, 'ttft_p95_s': 0.0888, 'latency_p95_s': 1.6295, 'errors': 0, 'ok': 20, 'wall_s': 25.476}
{'concurrency': 2, 'tokens_per_s': 158.67, 'ttft_p50_s': 0.0539, 'ttft_p95_s': 0.0919, 'latency_p95_s': 1.5811, 'errors': 0, 'ok': 20, 'wall_s': 12.623}
{'concurrency': 4, 'tokens_per_s': 275.39, 'ttft_p50_s': 0.0617, 'ttft_p95_s': 0.1214, 'latency_p95_s': 1.7514, 'errors': 0, 'ok': 20, 'wall_s': 7.273}
{'concurrency': 8, 'tokens_per_s': 454.94, 'ttft_p50_s': 0.1533, 'ttft_p95_s': 0.2076, 'latency_p95_s': 2.0367, 'errors': 0, 'ok': 20, 'wall_s': 4.403}
{'concurrency': 16, 'tokens_per_s': 683.82, 'ttft_p50_s': 0.246, 'ttft_p95_s': 0.2499, 'latency_p95_s': 2.4308, 'errors': 0, 'ok': 20, 'wall_s': 3.05}


In [57]:
def cost_per_million_tokens(tokens_per_s, gpu_hourly_usd):
    tokens_per_hour = tokens_per_s * 3600
    million_tokens_per_hour = tokens_per_hour / 1_000_000
    return round(gpu_hourly_usd / million_tokens_per_hour, 4)

GPU_HOURLY_USD = 0.35   # a representative on-demand T4-class price; swap in your real rate

for L in levels:
    L["cost_per_million_tokens_usd"] = cost_per_million_tokens(L["tokens_per_s"], GPU_HOURLY_USD)

for L in levels:
    print(f"c={L['concurrency']:>2}  tok/s={L['tokens_per_s']:>7.1f}  "
          f"p95={L['latency_p95_s']:.2f}s  $/M tok=${L['cost_per_million_tokens_usd']}")

c= 1  tok/s=   78.6  p95=1.63s  $/M tok=$1.2366
c= 2  tok/s=  158.7  p95=1.58s  $/M tok=$0.6127
c= 4  tok/s=  275.4  p95=1.75s  $/M tok=$0.353
c= 8  tok/s=  454.9  p95=2.04s  $/M tok=$0.2137
c=16  tok/s=  683.8  p95=2.43s  $/M tok=$0.1422


In [58]:
TARGET_P95_S = 2.0   # your SLO from this afternoon's prediction card

under_target = [L for L in levels if L["latency_p95_s"] <= TARGET_P95_S]
knee = max(under_target, key=lambda L: L["concurrency"]) if under_target else None
print("knee:", knee)

past_knee = [L for L in levels if knee and L["concurrency"] > knee["concurrency"]]
if past_knee:
    cheapest_past_knee = min(past_knee, key=lambda L: L["cost_per_million_tokens_usd"])
    print("cheapest $/M token level past the knee (SLO-violating):", cheapest_past_knee)
    print("-> cheaper on paper, but its p95 already exceeds your SLO -- "
          "not real usable capacity at your target.")

knee: {'concurrency': 4, 'tokens_per_s': 275.39, 'ttft_p50_s': 0.0617, 'ttft_p95_s': 0.1214, 'latency_p95_s': 1.7514, 'errors': 0, 'ok': 20, 'wall_s': 7.273, 'cost_per_million_tokens_usd': 0.353}
cheapest $/M token level past the knee (SLO-violating): {'concurrency': 16, 'tokens_per_s': 683.82, 'ttft_p50_s': 0.246, 'ttft_p95_s': 0.2499, 'latency_p95_s': 2.4308, 'errors': 0, 'ok': 20, 'wall_s': 3.05, 'cost_per_million_tokens_usd': 0.1422}
-> cheaper on paper, but its p95 already exceeds your SLO -- not real usable capacity at your target.


In [59]:
import math

def replicas_needed(required_tokens_per_s, knee_tokens_per_s):
    return math.ceil(required_tokens_per_s / knee_tokens_per_s)

def scale_out_cost(required_tokens_per_s, knee, gpu_hourly_usd):
    n = replicas_needed(required_tokens_per_s, knee["tokens_per_s"])
    return {
        "required_tokens_per_s": required_tokens_per_s,
        "replicas_needed": n,
        "total_hourly_cost_usd": round(n * gpu_hourly_usd, 2),
        "effective_p95_s": knee["latency_p95_s"],   # every replica runs at the same safe knee
    }

targets = [knee["tokens_per_s"] * m for m in (1.0, 1.5, 2.0, 3.0)]
scale_plan = [scale_out_cost(t, knee, GPU_HOURLY_USD) for t in targets]
for row in scale_plan:
    print(row)

{'required_tokens_per_s': 275.39, 'replicas_needed': 1, 'total_hourly_cost_usd': 0.35, 'effective_p95_s': 1.7514}
{'required_tokens_per_s': 413.085, 'replicas_needed': 2, 'total_hourly_cost_usd': 0.7, 'effective_p95_s': 1.7514}
{'required_tokens_per_s': 550.78, 'replicas_needed': 2, 'total_hourly_cost_usd': 0.7, 'effective_p95_s': 1.7514}
{'required_tokens_per_s': 826.17, 'replicas_needed': 3, 'total_hourly_cost_usd': 1.05, 'effective_p95_s': 1.7514}


In [60]:
report = {
    "gpu_hourly_usd": GPU_HOURLY_USD,
    "target_p95_s": TARGET_P95_S,
    "levels": levels,
    "knee": knee,
    "scale_out_plan": scale_plan,
}
with open("cost_report.json", "w") as f:
    json.dump(report, f, indent=2)
print(json.dumps(report, indent=2))

{
  "gpu_hourly_usd": 0.35,
  "target_p95_s": 2.0,
  "levels": [
    {
      "concurrency": 1,
      "tokens_per_s": 78.62,
      "ttft_p50_s": 0.0532,
      "ttft_p95_s": 0.0888,
      "latency_p95_s": 1.6295,
      "errors": 0,
      "ok": 20,
      "wall_s": 25.476,
      "cost_per_million_tokens_usd": 1.2366
    },
    {
      "concurrency": 2,
      "tokens_per_s": 158.67,
      "ttft_p50_s": 0.0539,
      "ttft_p95_s": 0.0919,
      "latency_p95_s": 1.5811,
      "errors": 0,
      "ok": 20,
      "wall_s": 12.623,
      "cost_per_million_tokens_usd": 0.6127
    },
    {
      "concurrency": 4,
      "tokens_per_s": 275.39,
      "ttft_p50_s": 0.0617,
      "ttft_p95_s": 0.1214,
      "latency_p95_s": 1.7514,
      "errors": 0,
      "ok": 20,
      "wall_s": 7.273,
      "cost_per_million_tokens_usd": 0.353
    },
    {
      "concurrency": 8,
      "tokens_per_s": 454.94,
      "ttft_p50_s": 0.1533,
      "ttft_p95_s": 0.2076,
      "latency_p95_s": 2.0367,
      "errors": 0,
 

In [61]:

# Green-check verifier for Lab W3D5 (benchmark harness).
# Paste this as the last cell of your day-5 notebook and run it. It reads
# bench_report.json (from the harness) and capacity-note.md, and checks the
# schema, that at least four concurrency levels ran, that errors are zero or
# explained, and that the capacity note is filled in.
#
# Last line is exactly one of:
#   GREEN CHECK: PASS
#   GREEN CHECK: FAIL (<reason>)
# No interactivity, no arguments; exit code matches.

import json, os, re

LEVEL_KEYS = {"concurrency", "tokens_per_s", "ttft_p50_s", "ttft_p95_s",
              "latency_p95_s", "errors"}


class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""


def fail(reason: str) -> "NoReturn":
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()


def main() -> None:
    # 1) bench report
    if not os.path.exists("bench_report.json"):
        fail("bench_report.json not found; run the harness in Cell 3")
    try:
        with open("bench_report.json") as fh:
            document = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"bench_report.json is not valid JSON: {exc}")

    # bench.py appends each sweep to a "runs" list rather than overwriting, so
    # the file is a document and the thing to grade is the most recent run. A
    # bare list is also accepted, for a report assembled by hand.
    if isinstance(document, dict) and isinstance(document.get("runs"), list):
        if not document["runs"]:
            fail("bench_report.json has no runs; the harness wrote nothing")
        levels = document["runs"][-1].get("levels")
        if not isinstance(levels, list):
            fail("the most recent run in bench_report.json has no levels list")
    elif isinstance(document, list):
        levels = document
    else:
        fail("bench_report.json must be the harness output ({'runs': [...]}) "
             "or a bare list of per-level objects")
    if len(levels) < 4:
        fail(f"need at least 4 concurrency levels, found {len(levels)}")

    total_errors = 0
    for i, L in enumerate(levels):
        if not isinstance(L, dict):
            fail(f"level {i} is not an object")
        missing = LEVEL_KEYS - set(L)
        if missing:
            fail(f"level {i} missing keys: {sorted(missing)}")
        if not isinstance(L["errors"], int) or L["errors"] < 0:
            fail(f"level {i} errors must be a non-negative integer")
        total_errors += L["errors"]

    # 2) the knee file from Cell 5
    if not os.path.exists("knee.json"):
        fail("knee.json not found; write it in Cell 5")
    try:
        with open("knee.json") as fh:
            knee = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"knee.json is not valid JSON: {exc}")
    target = knee.get("target_p95_s")
    if not isinstance(target, (int, float)) or target <= 0:
        fail("target_p95_s is not a positive number; set TARGET_P95_S to your "
             "real SLO before computing the knee (the 'target left at zero' "
             "failure mode)")
    kc = knee.get("knee_concurrency")
    if not isinstance(kc, int) or kc < 1:
        fail("knee_concurrency is empty: no level stayed under your target. "
             "Either your SLO is stricter than this stack can serve (explain "
             "that in the note) or the target was never set from the card")

    # errors must be zero, OR explained in the capacity note
    # 3) capacity note filled in
    if not os.path.exists("capacity-note.md"):
        fail("capacity-note.md not found")
    with open("capacity-note.md") as fh:
        note = fh.read()
    remaining = re.findall(r"FILL:", note)
    if remaining:
        fail(f"capacity-note.md has {len(remaining)} unfilled FILL: placeholders")

    if total_errors > 0 and not re.search(r"error", note, re.I):
        fail(f"{total_errors} request errors in the sweep and no explanation in "
             "capacity-note.md; zero errors, or explain them")

    # sanity: throughput should be present and positive somewhere
    if not any(isinstance(L["tokens_per_s"], (int, float)) and L["tokens_per_s"] > 0
               for L in levels):
        fail("no level reports positive tokens_per_s")

    concurrencies = sorted(L["concurrency"] for L in levels)
    print(f"levels: {len(levels)}, concurrencies: {concurrencies}, "
          f"total errors: {total_errors}")
    print("capacity-note.md: all fields filled")
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    # A notebook cell cannot exit nonzero without printing a red traceback over
    # the result line, so only signal by exit code when run as a plain script.
    try:
        get_ipython()  # defined only inside IPython/Colab
    except NameError:
        raise SystemExit(1)


levels: 5, concurrencies: [1, 2, 4, 8, 16], total errors: 0
capacity-note.md: all fields filled
GREEN CHECK: PASS
